<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/WHSAT_text_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets scikit-learn torch

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/main/cleaned_safety_data_set_A.csv")
# Keep only necessary columns
data_set = df[['description', 'severity', 'critical_risk']].dropna()
data_set.head()

In [ ]:
data_set['critical_risk_label'] = data_set['critical_risk'].astype('category').cat.codes
risk_labels = dict(enumerate(data_set['critical_risk'].astype('category').cat.categories))
print(risk_labels)

In [8]:
#to make 0–5
data_set['severity_label'] = data_set['severity'] - 1

In [9]:
#splitting the data set as trainning category (80%) and testing category (20%)
train_df, test_df = train_test_split(data_set, test_size=0.2, stratify=data_set['severity_label'], random_state=42)

In [10]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )

In [11]:
from torch.utils.data import Dataset

class SafetyDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize(texts)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [12]:
train_dataset = SafetyDataset(train_df['description'], train_df['critical_risk_label'])
test_dataset = SafetyDataset(test_df['description'], test_df['critical_risk_label'])

num_labels = data_set['critical_risk_label'].nunique()

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

In [14]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,          # keep small for CPU
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir='./logs',
    save_strategy="no"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1": f1_score(labels, preds, average='macro')
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [17]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

save_dir = "/content/drive/MyDrive/WHSAT/"
os.makedirs(save_dir, exist_ok=True)

# Get predictions from the trained model
predictions = trainer.predict(test_dataset)
preds = predictions.predictions
y_test = predictions.label_ids

np.save(os.path.join(save_dir, "transformer_probs.npy"), preds)
np.save(os.path.join(save_dir, "y_test.npy"), y_test)
np.save(os.path.join(save_dir, "test_indices.npy"), test_df.index.values)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
